In [ ]:
# Cell 1 — Install (restart kernel after this completes)
!pip install "trl==0.15.2" "peft==0.14.0" "accelerate==1.2.1" "datasets==3.5.0" bitsandbytes requests matplotlib -q
print("Done. Restart kernel now, then run Cell 2 onward.")

In [ ]:
# Cell 2 — Auth + torchao patch
# MUST run before any other cell — patches out broken torchao before transformers loads
import sys, importlib.util as _ilu, builtins, os

# Block torchao: makes is_torchao_available() return False everywhere in transformers
_orig_find_spec = _ilu.find_spec
def _no_torchao(name, package=None):
    if isinstance(name, str) and "torchao" in name:
        return None
    return _orig_find_spec(name, package)
_ilu.find_spec = _no_torchao
print("torchao patched out.")

hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    print("WARNING: HF_TOKEN not set — add it as a Space secret named HF_TOKEN")
else:
    from huggingface_hub import login
    login(token=hf_token)
    print("Logged in to HuggingFace Hub.")

builtins.hf_token = hf_token

In [ ]:
# Cell 3 — Config
import os, builtins

SPACE_URL         = "https://Bharath-1608-social-agent-negotiation-v1.hf.space"
TRAIN_TASKS       = ["single-round-consensus", "adversarial-information", "opioid-overdose"]
MODEL_NAME        = "Qwen/Qwen2.5-0.5B-Instruct"
HF_REPO           = "Bharath-1608/negotiation-agent-grpo"
HF_TOKEN          = getattr(builtins, "hf_token", os.environ.get("HF_TOKEN", ""))

N_EPOCHS          = 3
EPISODES_PER_TASK = 10
GROUP_SIZE        = 4
MAX_TURNS         = 20
LORA_R            = 16
LORA_ALPHA        = 32

VALID_ACTIONS = [
    "share_information", "propose_consensus", "accept_consensus",
    "reject_consensus", "challenge_proposal", "request_clarification",
    "flag_bias", "flag_agenda",
]
MEDICAL_KEYWORDS = [
    "patient", "diagnosis", "treatment", "medication", "dose",
    "symptoms", "vitals", "triage", "consensus", "evidence",
    "protocol", "critical", "stable", "urgent", "contraindication",
    "allergy", "imaging", "labs", "monitor", "consult",
]

print(f"Model          : {MODEL_NAME}")
print(f"Epochs         : {N_EPOCHS}  |  Episodes/task: {EPISODES_PER_TASK}  |  Max turns: {MAX_TURNS}")
print(f"Tasks          : {TRAIN_TASKS}")
print(f"HF_TOKEN set   : {bool(HF_TOKEN)}")

In [ ]:
# Cell 4 — Model load (bitsandbytes 4-bit + PEFT LoRA, no unsloth)
import os, builtins, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from huggingface_hub import login

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_R     = 16
LORA_ALPHA = 32
HF_TOKEN   = getattr(builtins, "hf_token", os.environ.get("HF_TOKEN", ""))

if HF_TOKEN:
    login(token=HF_TOKEN)

print(f"Loading {MODEL_NAME} in 4-bit...")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_use_double_quant = True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_cfg,
    device_map          = "auto",
    token               = HF_TOKEN or None,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_cfg = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type      = TaskType.CAUSAL_LM,
    bias           = "none",
)
model = get_peft_model(model, lora_cfg)

# Qwen requires left-padding
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}  ({100*trainable/total:.2f}%)")
print(f"Total    : {total:,}")
print("Model ready.")

In [ ]:
# Cell 5 — Rollout function
import torch, requests, json, re

SPACE_URL     = "https://Bharath-1608-social-agent-negotiation-v1.hf.space"
MAX_TURNS     = 20
VALID_ACTIONS = [
    "share_information", "propose_consensus", "accept_consensus",
    "reject_consensus", "challenge_proposal", "request_clarification",
    "flag_bias", "flag_agenda",
]
SYSTEM_PROMPT = (
    "You are a doctor AI agent negotiating a medical decision with another doctor AI.\n"
    "Output ONLY valid JSON. No markdown, no explanation.\n\n"
    "Standard actions:\n"
    '{"action_type":"<action>","content":"<msg>","reasoning":"<why>"}\n\n'
    "flag_bias:\n"
    '{"action_type":"flag_bias","content":"<msg>","reasoning":"<why>",'
    '"bias_location":"<where>","bias_direction":"<toward>","bias_correction":"<fix>"}\n\n'
    "flag_agenda:\n"
    '{"action_type":"flag_agenda","content":"<msg>","reasoning":"<why>",'
    '"agenda_type":"<cost_cutter or aggressive_treater>",'
    '"agenda_evidence":"<evidence>","agenda_counter":"<counter>"}'
)
FALLBACK = {
    "action_type":"share_information",
    "content":"Based on my private information, I believe we need to discuss the patient clinical findings and reach a consensus on the correct treatment approach.",
    "reasoning":"Sharing clinical data to progress toward a medically correct joint decision.",
}
FLAG_DEFAULTS = {
    "bias_location":"not specified","bias_direction":"not specified","bias_correction":"not specified",
    "agenda_type":"cost_cutter","agenda_evidence":"not specified","agenda_counter":"not specified",
}

def _parse(text):
    try:
        p = json.loads(text)
        if isinstance(p, dict): return p
    except Exception: pass
    try:
        m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
        if m:
            p = json.loads(m.group())
            if isinstance(p, dict): return p
    except Exception: pass
    return None

def rollout_episode(model, tokenizer, task_id):
    all_prompts, all_completions, history, session_id = [], [], [], None
    try:
        r = requests.post(f"{SPACE_URL}/reset", json={"task_id":task_id}, timeout=30)
        r.raise_for_status()
        state = r.json(); session_id = state.get("session_id")
    except Exception as e:
        print(f"  Reset failed {task_id}: {e}"); return [], [], 0.05
    obs_a, obs_b = state.get("obs_agent_a",{}), state.get("obs_agent_b",{})

    for turn in range(MAX_TURNS):
        agent_id = "agent_a" if turn%2==0 else "agent_b"
        obs      = obs_a if agent_id=="agent_a" else obs_b
        hist_str = "".join(f"  [{m['agent_id']}] {m['action_type']}: {m['content'][:200]}\n" for m in history[-6:]) or "  (No messages yet.)"
        prompt   = (SYSTEM_PROMPT+"\n\n"
            f"Task: {obs.get('task_description','Medical negotiation.')}\n"
            f"Phase: {obs.get('current_phase','triage')} (turn {obs.get('phase_turn',0)})\n"
            f"Private info:\n{json.dumps(obs.get('private_information',{}),indent=2)[:600]}\n\n"
            f"Conversation:\n{hist_str}\n"
            f"Available: {obs.get('available_actions', VALID_ACTIONS)}\n"
            f"Your turn as {agent_id}. JSON only.")
        try:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                    temperature=0.7, top_p=0.9, pad_token_id=tokenizer.pad_token_id)
            completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        except Exception as e:
            print(f"  Gen t{turn}: {e}"); completion = json.dumps(FALLBACK)
        all_prompts.append(prompt); all_completions.append(completion)
        parsed = _parse(completion)
        action = parsed if (parsed and parsed.get("action_type") in VALID_ACTIONS) else dict(FALLBACK)
        avail  = obs.get("available_actions", VALID_ACTIONS)
        if action["action_type"] not in avail:
            action["action_type"] = "share_information" if "share_information" in avail else (avail[0] if avail else "share_information")
        payload = {"session_id":session_id,"action":{
            "agent_id":agent_id,"action_type":action["action_type"],
            "content":str(action.get("content",FALLBACK["content"]))[:1000],
            "reasoning":str(action.get("reasoning",FALLBACK["reasoning"]))[:500],
        }}
        if action["action_type"]=="flag_bias":
            for f in ("bias_location","bias_direction","bias_correction"):
                payload["action"][f]=str(action.get(f,FLAG_DEFAULTS[f]))[:300]
        if action["action_type"]=="flag_agenda":
            for f in ("agenda_type","agenda_evidence","agenda_counter"):
                payload["action"][f]=str(action.get(f,FLAG_DEFAULTS[f]))[:300]
        try:
            sr = requests.post(f"{SPACE_URL}/step", json=payload, timeout=30)
            if sr.status_code in (503,422): continue
            sr.raise_for_status(); step = sr.json()
        except Exception as e:
            print(f"  Step t{turn}: {e}"); continue
        obs_a,obs_b = step.get("obs_agent_a",obs_a), step.get("obs_agent_b",obs_b)
        history.append({"agent_id":agent_id,"action_type":action["action_type"],"content":action.get("content","")})
        if step.get("done"):
            ep = step.get("episode_result") or {}
            if ep.get("total_reward"): return all_prompts, all_completions, float(ep["total_reward"])
            try:
                s = requests.get(f"{SPACE_URL}/state", params={"session_id":session_id}, timeout=30).json()
                return all_prompts, all_completions, float(s.get("cumulative_reward",0.05))
            except Exception: return all_prompts, all_completions, 0.05
    try:
        s = requests.get(f"{SPACE_URL}/state", params={"session_id":session_id}, timeout=30).json()
        return all_prompts, all_completions, float(s.get("cumulative_reward",0.05))
    except Exception: return all_prompts, all_completions, 0.05

print("rollout_episode() ready.")
try:
    h = requests.get(f"{SPACE_URL}/health", timeout=15).json()
    print(f"Environment: {h.get('tasks_available')} tasks, reward range {h.get('reward_range')}")
except Exception as e: print(f"Cannot reach env: {e}")

In [ ]:
# Cell 6 — Reward function (deterministic, no API calls)
import json, re

VALID_ACTIONS    = ["share_information","propose_consensus","accept_consensus","reject_consensus","challenge_proposal","request_clarification","flag_bias","flag_agenda"]
MEDICAL_KEYWORDS = ["patient","diagnosis","treatment","medication","dose","symptoms","vitals","triage","consensus","evidence","protocol","critical","stable","urgent","contraindication","allergy","imaging","labs","monitor","consult"]
ADVERSARIAL      = {"flag_agenda","flag_bias","challenge_proposal"}

def compute_rewards(prompts, completions, task_ids=None, **kwargs):
    rewards = []
    for c in completions:
        score, parsed = 0.0, None
        try: parsed = json.loads(c)
        except Exception: pass
        if parsed is None:
            try:
                m = re.search(r"\{[^{}]*\}", c, re.DOTALL)
                if m: parsed = json.loads(m.group())
            except Exception: pass
        if isinstance(parsed, dict):
            score += 0.3
            atype, content, reason = parsed.get("action_type",""), parsed.get("content",""), parsed.get("reasoning","")
            if atype in VALID_ACTIONS:                             score += 0.2
            if len(str(content))  > 50:                           score += 0.2
            if len(str(reason))   > 30:                           score += 0.2
            if any(k in str(content).lower() for k in MEDICAL_KEYWORDS): score += 0.1
            if atype in ADVERSARIAL:                              score = min(1.0, score+0.15)
        rewards.append(round(min(1.0, max(0.0, score)), 4))
    return rewards

_t = [
    '{"action_type":"share_information","content":"The patient shows critical symptoms including elevated troponin and hypotension requiring immediate triage decision.","reasoning":"Sharing lab results to reach correct consensus."}',
    'not json',
    '{"action_type":"flag_agenda","content":"I am flagging a cost-cutting institutional bias here — patient welfare must override the financial mandate in this critical triage decision.","reasoning":"Detecting agenda before consensus."}',
]
_s = compute_rewards(None, _t)
assert _s[0]==1.0 and _s[1]==0.0 and _s[2]==1.0, f"Smoke test failed: {_s}"
print(f"compute_rewards() ready. Smoke: {_s}")

In [ ]:
# Cell 7 — GRPO training (self-contained)
import os, builtins, json, re, torch, requests
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

SPACE_URL         = "https://Bharath-1608-social-agent-negotiation-v1.hf.space"
TRAIN_TASKS       = ["single-round-consensus", "adversarial-information", "opioid-overdose"]
N_EPOCHS          = 3
EPISODES_PER_TASK = 10
GROUP_SIZE        = 4
MAX_TURNS         = 20
VALID_ACTIONS     = ["share_information","propose_consensus","accept_consensus","reject_consensus","challenge_proposal","request_clarification","flag_bias","flag_agenda"]
MEDICAL_KEYWORDS  = ["patient","diagnosis","treatment","medication","dose","symptoms","vitals","triage","consensus","evidence","protocol","critical","stable","urgent","contraindication","allergy","imaging","labs","monitor","consult"]
ADVERSARIAL       = {"flag_agenda","flag_bias","challenge_proposal"}
FALLBACK          = {"action_type":"share_information","content":"Based on my private information, I believe we need to discuss the patient clinical findings and reach a consensus on the correct treatment approach.","reasoning":"Sharing clinical data to progress toward a medically correct joint decision."}
FLAG_DEFAULTS     = {"bias_location":"not specified","bias_direction":"not specified","bias_correction":"not specified","agenda_type":"cost_cutter","agenda_evidence":"not specified","agenda_counter":"not specified"}
SYSTEM_PROMPT     = ("You are a doctor AI agent negotiating a medical decision with another doctor AI.\nOutput ONLY valid JSON.\n\n"
    'Standard: {"action_type":"<action>","content":"<msg>","reasoning":"<why>"}\n'
    'flag_bias: {"action_type":"flag_bias","content":"<msg>","reasoning":"<why>","bias_location":"<w>","bias_direction":"<d>","bias_correction":"<fix>"}\n'
    'flag_agenda: {"action_type":"flag_agenda","content":"<msg>","reasoning":"<why>","agenda_type":"<cost_cutter or aggressive_treater>","agenda_evidence":"<ev>","agenda_counter":"<ct>"}')

def _rw(prompts, completions, task_ids=None, **kw):
    out = []
    for c in completions:
        s, p = 0.0, None
        try: p = json.loads(c)
        except Exception: pass
        if p is None:
            try:
                m = re.search(r"\{[^{}]*\}", c, re.DOTALL)
                if m: p = json.loads(m.group())
            except Exception: pass
        if isinstance(p, dict):
            s += 0.3; at = p.get("action_type",""); ct = p.get("content",""); rs = p.get("reasoning","")
            if at in VALID_ACTIONS: s += 0.2
            if len(str(ct))>50: s += 0.2
            if len(str(rs))>30: s += 0.2
            if any(k in str(ct).lower() for k in MEDICAL_KEYWORDS): s += 0.1
            if at in ADVERSARIAL: s = min(1.0, s+0.15)
        out.append(round(min(1.0,max(0.0,s)),4))
    return out

def _parse(text):
    try:
        p = json.loads(text)
        if isinstance(p,dict): return p
    except Exception: pass
    try:
        m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
        if m:
            p = json.loads(m.group())
            if isinstance(p,dict): return p
    except Exception: pass
    return None

def _roll(model, tokenizer, task_id):
    ps, cs, hist, sid = [], [], [], None
    try:
        r = requests.post(f"{SPACE_URL}/reset", json={"task_id":task_id}, timeout=30)
        r.raise_for_status(); st=r.json(); sid=st.get("session_id")
    except Exception as e:
        print(f"  Reset {task_id}: {e}"); return [], [], 0.05
    oa, ob = st.get("obs_agent_a",{}), st.get("obs_agent_b",{})
    for turn in range(MAX_TURNS):
        aid = "agent_a" if turn%2==0 else "agent_b"
        obs = oa if aid=="agent_a" else ob
        hs  = "".join(f"  [{m['agent_id']}] {m['action_type']}: {m['content'][:200]}\n" for m in hist[-6:]) or "  (No messages yet.)"
        pr  = (SYSTEM_PROMPT+f"\nTask:{obs.get('task_description','')}\nPhase:{obs.get('current_phase','triage')}(t{obs.get('phase_turn',0)})\n"
               f"Info:\n{json.dumps(obs.get('private_information',{}),indent=2)[:600]}\nConvo:\n{hs}\nAvail:{obs.get('available_actions',VALID_ACTIONS)}\n{aid} JSON only.")
        try:
            inp = tokenizer(pr, return_tensors="pt", truncation=True, max_length=1800).to(model.device)
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=256, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tokenizer.pad_token_id)
            comp = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        except Exception as e:
            print(f"  Gen t{turn}: {e}"); comp = json.dumps(FALLBACK)
        ps.append(pr); cs.append(comp)
        parsed = _parse(comp); action = parsed if (parsed and parsed.get("action_type") in VALID_ACTIONS) else dict(FALLBACK)
        avail = obs.get("available_actions", VALID_ACTIONS)
        if action["action_type"] not in avail:
            action["action_type"] = "share_information" if "share_information" in avail else (avail[0] if avail else "share_information")
        pl = {"session_id":sid,"action":{"agent_id":aid,"action_type":action["action_type"],"content":str(action.get("content",FALLBACK["content"]))[:1000],"reasoning":str(action.get("reasoning",FALLBACK["reasoning"]))[:500]}}
        if action["action_type"]=="flag_bias":
            for f in ("bias_location","bias_direction","bias_correction"): pl["action"][f]=str(action.get(f,FLAG_DEFAULTS[f]))[:300]
        if action["action_type"]=="flag_agenda":
            for f in ("agenda_type","agenda_evidence","agenda_counter"): pl["action"][f]=str(action.get(f,FLAG_DEFAULTS[f]))[:300]
        try:
            sr = requests.post(f"{SPACE_URL}/step", json=pl, timeout=30)
            if sr.status_code in (503,422): continue
            sr.raise_for_status(); step=sr.json()
        except Exception as e:
            print(f"  Step t{turn}: {e}"); continue
        oa,ob = step.get("obs_agent_a",oa), step.get("obs_agent_b",ob)
        hist.append({"agent_id":aid,"action_type":action["action_type"],"content":action.get("content","")})
        if step.get("done"):
            ep = step.get("episode_result") or {}
            if ep.get("total_reward"): return ps, cs, float(ep["total_reward"])
            try:
                s = requests.get(f"{SPACE_URL}/state", params={"session_id":sid}, timeout=30).json()
                return ps, cs, float(s.get("cumulative_reward",0.05))
            except Exception: return ps, cs, 0.05
    try:
        s = requests.get(f"{SPACE_URL}/state", params={"session_id":sid}, timeout=30).json()
        return ps, cs, float(s.get("cumulative_reward",0.05))
    except Exception: return ps, cs, 0.05

rewards_log, all_p, all_t = {t:[] for t in TRAIN_TASKS}, [], []
model.eval()
print("="*55+f"\nPHASE 1 — Collecting episodes\n"+"="*55)
for task_id in TRAIN_TASKS:
    print(f"\n  Task: {task_id}")
    for ep in range(EPISODES_PER_TASK):
        prompts, comps, reward = _roll(model, tokenizer, task_id)
        rewards_log[task_id].append(reward); all_p.extend(prompts); all_t.extend([task_id]*len(prompts))
        print(f"    ep{ep+1:02d}/{EPISODES_PER_TASK} reward={reward:.4f} steps={len(prompts)}")

if not all_p:
    print("WARNING: No prompts — fallback"); all_p=["You are a doctor. Triage STEMI. Respond JSON: {action_type, content, reasoning}"]*16; all_t=["single-round-consensus"]*16
dataset = Dataset.from_dict({"prompt":all_p, "task_ids":all_t})
print(f"\nDataset: {len(dataset)} samples")

config = GRPOConfig(
    output_dir                  = "./grpo_output",
    num_train_epochs            = N_EPOCHS,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.05,
    max_completion_length       = 256,
    num_generations             = GROUP_SIZE,
    logging_steps               = 1,
    save_steps                  = 50,
    report_to                   = "none",
)

model.train()
trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [_rw],
    args             = config,
    train_dataset    = dataset,
)
print("\n"+"="*55+f"\nPHASE 2 — GRPO Training\n"+"="*55)
trainer.train()
print("\nTraining complete!")
for tid, sc in rewards_log.items():
    if sc: print(f"  {tid:<35} avg={sum(sc)/len(sc):.4f}  best={max(sc):.4f}")

In [ ]:
# Cell 8 — Plot reward curve
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, numpy as np

COLORS = {"single-round-consensus":"#6366f1","adversarial-information":"#ef4444","opioid-overdose":"#22c55e"}
fig, ax = plt.subplots(figsize=(12,6))
fig.patch.set_facecolor("#0d1117"); ax.set_facecolor("#0d1117")
has_data = False
for tid, scores in rewards_log.items():
    if not scores: continue
    has_data = True; color = COLORS.get(tid,"#94a3b8"); eps = list(range(1,len(scores)+1))
    ax.plot(eps, scores, color=color, alpha=0.25, linewidth=1.2)
    ax.scatter(eps, scores, color=color, s=22, alpha=0.5, zorder=3)
    if len(scores)>=3:
        sm = np.convolve(scores, np.ones(3)/3, mode="valid")
        ax.plot(range(2,len(scores)+1), sm, color=color, linewidth=2.5, linestyle="--", label=f"{tid} (smoothed)", zorder=4)
    else:
        ax.plot(eps, scores, color=color, linewidth=2.5, linestyle="--", label=tid, zorder=4)
ax.set_xlabel("Episode",color="white",fontsize=13); ax.set_ylabel("Reward",color="white",fontsize=13)
ax.set_title("GRPO Training — Social Agent Negotiation",color="white",fontsize=14,fontweight="bold")
ax.set_ylim(0,1.05); ax.tick_params(colors="white")
ax.spines[["top","right"]].set_visible(False); ax.spines[["bottom","left"]].set_color("#333")
ax.yaxis.grid(True,color="#1e1e2e",linewidth=0.8); ax.set_axisbelow(True)
if has_data: ax.legend(facecolor="#161b22",edgecolor="#333",labelcolor="white",fontsize=11,loc="lower right")
plt.tight_layout(); plt.savefig("reward_curve_grpo.png",dpi=150,bbox_inches="tight",facecolor="#0d1117"); plt.show()
print("Saved: reward_curve_grpo.png")

In [ ]:
# Cell 9 — Save and push to Hub
import os, builtins, shutil
from huggingface_hub import HfApi

HF_TOKEN = getattr(builtins, "hf_token", os.environ.get("HF_TOKEN",""))
HF_REPO  = "Bharath-1608/negotiation-agent-grpo"
SAVE_DIR = "negotiation_lora_final"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
if os.path.exists("reward_curve_grpo.png"):
    shutil.copy("reward_curve_grpo.png", os.path.join(SAVE_DIR,"reward_curve_grpo.png"))

with open(os.path.join(SAVE_DIR,"README.md"),"w") as f:
    f.write("---\nbase_model: Qwen/Qwen2.5-0.5B-Instruct\ntags:\n  - negotiation\n  - grpo\n  - openenv\n---\n\n"
            "# Negotiation Agent GRPO (Qwen2.5-0.5B)\n\n"
            "Fine-tuned via GRPO on social-agent-negotiation-v1 OpenEnv environment.\n")

print(f"Saved to ./{SAVE_DIR}")
if not HF_TOKEN:
    print("HF_TOKEN not set — skipping push.")
else:
    api = HfApi()
    api.create_repo(HF_REPO, token=HF_TOKEN, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=SAVE_DIR, repo_id=HF_REPO, repo_type="model", token=HF_TOKEN)
    print(f"Pushed to https://huggingface.co/{HF_REPO}")